# Robustness & XAI (univariat) unter CMAPSS‑ähnlichen Störungen (CNN + RandomForest, Binary Classification)

Dieses Notebook ist für **dein eigenes Tabular-Setting** (Features wie `mittelwert_temp`, …; Label `fehler`), aber die **Störungen/Szenarien** sind **analog zu CMAPSS** aufgebaut:

## Szenarien (CMAPSS‑Style)
1. **baseline**: keine Störung (Train/Eval clean)
2. **covariance_shift**: Feature‑Skalierung auf Subset (nur **Eval** auf Shift; Modell bleibt clean)
3. **environmental_shift**: Feature‑Bias auf Subset (nur **Eval** auf Shift; Modell bleibt clean)
4. **model_drift**: zusätzliche Trainingsphase auf drifted Daten (CNN: weitertrainieren; RF: neu trainieren)
5. **catastrophic_forgetting**: Domain A train → Fine‑Tune/Train auf Domain B → Eval auf A und B
6. **underfitting / overfitting**: absichtlich zu simpel / zu komplex bzw. zu wenig Daten

## XAI pro Szenario & Modell
### CNN (Keras)
- Gradients (Batch‑Aggregat) via `tf.GradientTape` (robust, ohne TF‑Explain Abhängigkeit)
- optional: `tf-explain` Integrated Gradients / Vanilla / SmoothGrad (wenn installiert)
- PDP (global, p(y=1))
- LIME (lokal)
- DiCE (counterfactuals, desired_class='opposite') via sklearn‑Wrapper
- Surrogate Tree Rules

### RandomForest (sklearn Pipeline)
- PDP (global)
- LIME (lokal)
- DiCE (CFs) (mit robusten Checks)
- Surrogate Tree Rules

> **Wichtig:** Du musst unten nur `df_base`, `feature_cols`, `label_col` setzen und ggf. `prepare_df(...)` an deine Daten anpassen.


In [1]:
import pandas as pd
import numpy as np
#import seaborn as sns# z.B. für Themes oder Stil
import matplotlib
from scipy.stats import skew
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.metrics import mean_squared_error
from sklearn.utils.class_weight import compute_class_weight
#import tensorflow as tf
#tf.compat.v1.reset_default_graph()
#tf.compat.v1.disable_eager_execution()
#from tensorflow import keras
#from keras import layers

import tensorflow as tf
tf.compat.v1.reset_default_graph()
tf.compat.v1.disable_eager_execution()

from tensorflow.keras.models import Sequential
from tensorflow.keras import layers, Input
from tensorflow.keras import optimizers, metrics, callbacks
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.compat.v1.keras import backend as K   # <-- ja, korrekt

import os, glob, re
#import shap
#import shap.utils.transformers as shap_tr
#shap_tr.is_transformers_lm = lambda model: False  # Patch
from sklearn.pipeline import make_pipeline
import alibi
from alibi.explainers import AnchorTabular
from alibi.explainers import Counterfactual

from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from typing import Dict, Any, Optional, Tuple, List
#import transformers
#from tensorflow.compat.v1.keras import backend as K

C:\Users\ko\Anaconda3\envs\Diss_Alibi\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ============================================================
# 1) Config
# ============================================================
CONFIG = {
    "seed": 42,
    "id_col": "point_id",
    "local_point_ids": [398, 1996, 728],

    # Splits
    "train_frac": 0.7,
    "val_frac_within_train": 0.2,

    # CMAPSS-Style Störungen
    "cov_shift_scale": 1.3,
    "cov_shift_frac_features": 0.25,
    "env_shift_bias": 0.5,
    "env_shift_frac_features": 0.30,

    # Model drift
    "drift_extra_epochs": 20,   # CNN: weitertrainieren
    "rf_retrain_on_drift": True,

    # Catastrophic forgetting (Domain B = A + (cov+env shift))
    "forget_ft_epochs": 30,     # CNN fine-tune auf B
    "forget_rf_retrain": True,

    # Under/Overfitting
    "under_single_feature": None,   # z.B. "mittelwert_temp"
    "over_small_train_size": 500,

    # XAI
    # --- SHAP ---
    "shap_explainer": "auto",          # "auto" | "tree" | "kernel" | "deep"
    "shap_task": "classification",     # oder "regression"
    "shap_background_size_global": 1000,   # 1000 Hintergrundstichprobe (global)
    "shap_background_size_local": 200,     # Hintergrundstichprobe (lokal)
    "shap_max_display": 20,            # Top-N Features für globale Plots
    "shap_local_n_points": 20,         # wie viele lokale Punkte erklären (falls nicht über IDs vorgegeben)
    "shap_local_use_fixed_ids": False, # True, wenn du explizit IDs/Indices vorgibst
    "shap_local_id_col": "point_id",   # nur relevant wenn use_fixed_ids=True
    # optional: "shap_local_ids": [...],  # Liste von point_ids / Indices, wenn du fix erklären willst

    # KernelSHAP (nur falls explainer="kernel" bzw. nötig)
    "shap_kernel_nsamples": 200,       # Laufzeit/Qualität Tradeoff
    "shap_kernel_l1_reg": "num_features(10)",

    # --- Saliency Heatmap ---
    "saliency_method": "integrated_gradients",  # "vanilla_gradients" | "integrated_gradients" | "smoothgrad"
    "saliency_target_class": 1,          # bei binärer Klassifikation meist 1
    "saliency_normalize": "minmax",      # "minmax" | "zscore" | None
    "saliency_aggregate": "abs",         # "abs" | "raw" | "pos" (für Heatmap-Aufbereitung)
    "saliency_ig_steps": 32,             # nur IG
    "saliency_smoothgrad_samples": 20,   # nur SmoothGrad
    "saliency_smoothgrad_noise_sigma": 0.1,
    # in CONFIG["xai"] oder flach:
    "saliency_context_k": 5,              # +/- 5 Punkte um die ID
    "saliency_max_display_features": 40,  # Top-K Features pro Plot
    "saliency_max_fig_width": 18,
    "saliency_max_fig_height": 10,


    # --- Prototypes & Criticisms ---
    "proto_method": "kmedoids",          # "kmedoids" (robust, ohne Extra-Libs) oder "random"
    "proto_n_prototypes": 10,
    "proto_n_criticisms": 10,
    "proto_distance": "euclidean",       # "euclidean" | "cosine"
    "proto_candidate_pool_size": 5000,   # max. Kandidaten (subsample) für Geschwindigkeit
    "proto_use_stratified": True,        # bei Klassifikation pro Klasse/Label stratifizieren

    # Criticisms (wenn du MMD-/Kernel-basiert machst)
    "critic_kernel": "rbf",              # "rbf" | "linear"
    "critic_gamma": "scale",             # "scale" | float
}

# Compat: einige Blöcke erwarten fixed_local_ilocs/fixed_local_ids (wie patched_v02)
CONFIG.setdefault("fixed_local_ilocs", CONFIG.get("local_point_ids", [398, 1996, 728]))
CONFIG.setdefault("fixed_local_ids", None)

def ensure_point_id(df: pd.DataFrame, id_col: str = "point_id") -> pd.DataFrame:
    """Adds a stable integer id column if missing. Keeps row order."""
    df = df.copy()
    if id_col not in df.columns:
        df[id_col] = np.arange(len(df), dtype=int)
    return df


def set_seeds(seed: int = 42):
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_seeds(CONFIG["seed"])


In [3]:
# <-- Dein Ordner (laut Ausgabe)
pfad = r"C:\User\OneDrive - fir.rwth-aachen.de\Diss\99_Schreibversionen\Model 4\Daten Fallstudien\Cm4B2C\pressure"

dateien = glob.glob(os.path.join(pfad, "*.txt"))
assert len(dateien) > 0, "Keine Dateien gefunden – Pfad/Endung prüfen."

alle = []
for datei in dateien:
    # 1) Roh einlesen, am ' - ' trennen
    df = pd.read_csv(
        datei,
        sep=r"\s-\s",            # Trennt exakt am " - "
        engine="python",
        header=None,
        names=["zeit", "druck_raw"],
        dtype=str,
        on_bad_lines="skip"
    )

    # 2) Druck als Float aus "... PSI" extrahieren
    df["druck"] = (
        df["druck_raw"]
        .str.extract(r"([0-9]+(?:\.[0-9]+)?)")   # 362.69 aus "362.69 PSI"
        .astype(float)
    )

    # 3) Zeit parsen (deutsches Datum wie 10.7.2025, 12:29:01)
    df["zeit"] = pd.to_datetime(df["zeit"], format="%d.%m.%Y, %H:%M:%S", errors="coerce")

    # 4) Aufräumen & Quelle anhängen
    df = df.dropna(subset=["zeit", "druck"]).copy()
    df["quelle"] = os.path.basename(datei)
    alle.append(df[["zeit", "druck", "quelle"]])

# Zusammenführen
df_press = pd.concat(alle, ignore_index=True)

# Druckfehler markieren und zählen
df_press["druckfehler"] = df_press["druck"] < 340

df_press
# Optional: speichern
# df_press.to_csv(os.path.join(pfad, "pressure_parsed.csv"), index=False)
df = df_press.copy()     # oder: df = df_temp.copy()
wert_col = "druck"       # oder "temp"

# --- 1️⃣ Zeitstempel auf volle Minute runden ---
df["minute"] = df["zeit"].dt.floor("T")

# --- 2️⃣ Gruppieren nach Minute ---
gruppen = df.groupby("minute")[wert_col]

# --- 3️⃣ Kennzahlen berechnen ---
agg_df_press = gruppen.agg([
    "count",           # Anzahl Messungen pro Minute
    "mean",            # Mittelwert
    "median",          # Median
    "std",             # Standardabweichung
    "min", "max",      # Minimum / Maximum
    "skew"             # Schiefe (Skewness)
]).rename(columns={
    "count": "anzahl",
    "mean": "mittelwert",
    "median": "median",
    "std": "stdabw",
    "min": "minimum",
    "max": "maximum",
    "skew": "schiefe"
})

# --- 4️⃣ Quartile separat berechnen und anhängen ---
quartiles = gruppen.quantile([0.25, 0.75]).unstack()
agg_df_press["Q1"] = quartiles[0.25]
agg_df_press["Q3"] = quartiles[0.75]
agg_df_press["IQR"] = agg_df_press["Q3"] - agg_df_press["Q1"]

# --- 5️⃣ Optional: Druckfehler- oder Temperaturfehlerquote pro Minute ---
if "druckfehler" in df.columns:
    fehler = df.groupby("minute")["druckfehler"].mean()
    agg_df_press["fehlerquote"] = fehler * 100
elif "tempfehler" in df.columns:
    fehler = df.groupby("minute")["tempfehler"].mean()
    agg_df_press["fehlerquote"] = fehler * 100

# --- 6️⃣ Ergebnis ---
agg_df_press = agg_df_press.reset_index()
print(agg_df_press.head(10))

agg_df_press["druckfehler"] = (
    (agg_df_press["median"] < 340) |
    (agg_df_press["maximum"] > 400) |
    (agg_df_press["Q3"] > 370)
)

C:\Temp\ipykernel_24920\1863590040.py:48: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  df["minute"] = df["zeit"].dt.floor("T")


               minute  anzahl  mittelwert  median      stdabw  minimum  \
0 2025-07-10 12:29:00     117  278.481026   174.0  138.005851    174.0   
1 2025-07-10 12:30:00     120  285.851083   174.0  141.020684    174.0   
2 2025-07-10 12:31:00     120  275.051417   174.0  137.443918    174.0   
3 2025-07-10 12:32:00     120  280.717000   174.0  139.308206    174.0   
4 2025-07-10 12:33:00     119  289.290672   174.0  141.970082    174.0   
5 2025-07-10 12:34:00     120  277.164583   174.0  137.364221    174.0   
6 2025-07-10 12:35:00     120  276.211417   174.0  137.809784    174.0   
7 2025-07-10 12:36:00     120  281.007917   174.0  138.801365    174.0   
8 2025-07-10 12:37:00     119  282.189160   174.0  139.483394    174.0   
9 2025-07-10 12:38:00     120  279.272917   174.0  137.998185    174.0   

   maximum   schiefe     Q1        Q3       IQR  fehlerquote  
0    500.0  0.757341  174.0  439.5700  265.5700    66.666667  
1    500.0  0.649784  174.0  460.7825  286.7825    65.83333

## 2) Daten anschließen

**Du setzt hier deine Variablen:**
- `df_base`: DataFrame mit Features + Label
- `feature_cols`: Liste der Feature-Spalten
- `label_col`: Name der Label-Spalte (bei dir: `"druckfehler"`)  
- **Univariat:** wir verwenden genau **eine** Feature-Spalte (`single_feature`).


In [4]:
# ============================================================
# 2) DATA HOOKS: bitte anpassen
# ============================================================
df_base = agg_df_press   # <-- DEIN DataFrame

# exakt die Features, die du fürs NN nutzt
# Univariat: wähle GENAU EINE Feature-Spalte
single_feature = "mittelwert"   # <- ausgewählte Spalte
feature_cols = [single_feature]

# (Optional) Original-Liste (multivariat) – nur zur Referenz:
# feature_cols_multivariat = [
#     "anzahl", "mittelwert", "median", "stdabw", "minimum", "maximum",
#     "schiefe", "Q1", "Q3", "IQR", "fehlerquote"
# ]


label_col = "druckfehler"

# ============================================================
# STABILE POINT_ID – MUSS GENAU EINMAL AUSGEFÜHRT WERDEN
# ============================================================
def ensure_point_id(df: pd.DataFrame, id_col="point_id"):
    df = df.copy()
    if id_col not in df.columns:
        df[id_col] = np.arange(len(df), dtype=int)
    return df

def prepare_df(df: pd.DataFrame) -> pd.DataFrame:
    """Minimal-Preprocessing. Passe es bei Bedarf an."""
    df = df.copy()
    if label_col in df.columns:
        df[label_col] = df[label_col].astype(int)
    return df

df = ensure_point_id(df)

# ============================================================
# FESTE VERGLEICHSPUNKTE (iloc → point_id)
# ============================================================
CONFIG["local_point_ilocs"] = [398, 1996, 728]

In [5]:
# ============================================================
# 3) Splits + Scaler (A: clean)
# ============================================================
def split_time_ordered(df: pd.DataFrame, train_frac: float):
    """Zeit-/Reihenfolge-Split (kein Shuffle)."""
    df = df.copy()
    split_idx = int(len(df) * train_frac)
    df_train = df.iloc[:split_idx].copy()
    df_test  = df.iloc[split_idx:].copy()
    return df_train, df_test

def make_train_val(df_train: pd.DataFrame, val_frac: float):
    n = len(df_train)
    split_idx = int(n * (1.0 - val_frac))
    return df_train.iloc[:split_idx].copy(), df_train.iloc[split_idx:].copy()

def make_scaler_from_train(df_train: pd.DataFrame, feature_cols) -> StandardScaler:
    X = df_train[feature_cols].to_numpy().astype("float32")
    scaler = StandardScaler()
    scaler.fit(X)
    return scaler


In [6]:
# ============================================================
# 4) CMAPSS-Style Störungen auf DataFrame (Tabular)
# ============================================================
def _rng(seed: int):
    return np.random.RandomState(seed)

def apply_covariance_shift_df(df: pd.DataFrame, feature_cols, scale=1.3, frac_features=0.25, seed=42):
    rng = _rng(seed)
    df_s = df.copy()
    d = len(feature_cols)
    k = max(1, int(frac_features * d))
    idx = rng.choice(d, size=k, replace=False)
    cols = [feature_cols[i] for i in idx]
    df_s[cols] = df_s[cols] * float(scale)
    return df_s, {"type":"covariance_shift", "scale": float(scale), "cols": cols}

def apply_environmental_shift_df(df: pd.DataFrame, feature_cols, bias=0.5, frac_features=0.30, seed=42):
    rng = _rng(seed)
    df_s = df.copy()
    d = len(feature_cols)
    k = max(1, int(frac_features * d))
    idx = rng.choice(d, size=k, replace=False)
    cols = [feature_cols[i] for i in idx]
    df_s[cols] = df_s[cols] + float(bias)
    return df_s, {"type":"environmental_shift", "bias": float(bias), "cols": cols}

def make_domain_B_from_A(df: pd.DataFrame, feature_cols, seed=42):
    df_b, meta_cov = apply_covariance_shift_df(
        df, feature_cols,
        scale=CONFIG['cov_shift_scale'],
        frac_features=CONFIG['cov_shift_frac_features'],
        seed=seed+1
    )
    df_b, meta_env = apply_environmental_shift_df(
        df_b, feature_cols,
        bias=CONFIG['env_shift_bias'],
        frac_features=CONFIG['env_shift_frac_features'],
        seed=seed+2
    )
    return df_b, {"cov": meta_cov, "env": meta_env}

In [7]:
# ============================================================
# 5) Modelle: DenseNN (Keras) + SVM Pipeline
# ============================================================
# --- Keras Imports robust (tf.keras falls vorhanden, sonst standalone keras) ---
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")  # wichtig, falls Keras 3 installiert ist

try:
    from tensorflow.keras import layers, optimizers, metrics, callbacks
    from tensorflow.keras.models import Sequential
    KERAS_API = "tf.keras"
except ModuleNotFoundError:
    import keras
    from keras import layers, optimizers, metrics, callbacks
    from keras.models import Sequential
    KERAS_API = "keras"

print("✅ Using:", KERAS_API)


def get_class_weight_dict(y: np.ndarray) -> Dict[int, float]:
    """Balanced class weights (für Klassendrift/Imbalance)."""
    y = np.asarray(y).astype(int)
    classes = np.unique(y)
    cw = compute_class_weight(class_weight="balanced", classes=classes, y=y)
    return {int(c): float(w) for c, w in zip(classes, cw)}


# -------------------------
# DenseNN (Keras)
# -------------------------
def build_dense_model(n_features: int) -> tf.keras.Model:
    model = Sequential([
        layers.Input(shape=(n_features,)),
        layers.BatchNormalization(),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.2),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.2),
        layers.Dense(1, activation="sigmoid"),
    ])
    model.compile(
        optimizer=optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=[metrics.AUC(name="auc"), metrics.Precision(name="prec"), metrics.Recall(name="rec")],
    )
    return model


def train_dense_on_df(
    df_train: pd.DataFrame,
    scaler: StandardScaler,
    feature_cols,
    label_col: str,
    epochs: int = 200,
    batch_size: int = 32,
    seed: int = 42,
) -> tf.keras.Model:
    set_seeds(seed)

    X_raw = df_train[feature_cols].to_numpy().astype("float32")
    y = df_train[label_col].astype(int).to_numpy()

    X_scaled = scaler.transform(X_raw).astype("float32")

    class_weight_dict = get_class_weight_dict(y)

    model = build_dense_model(X_scaled.shape[1])
    cb = [
        callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=20, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(monitor="val_auc", mode="max", factor=0.5, patience=8, min_lr=1e-5),
    ]
    model.fit(
        X_scaled, y,
        epochs=epochs,
        batch_size=batch_size,
        validation_split=0.2,
        class_weight=class_weight_dict,
        callbacks=cb,
        verbose=0,
    )
    return model


def continue_train_dense(
    model: tf.keras.Model,
    df_train: pd.DataFrame,
    scaler: StandardScaler,
    feature_cols,
    label_col: str,
    extra_epochs: int,
    batch_size: int = 32,
):
    X_raw = df_train[feature_cols].to_numpy().astype("float32")
    y = df_train[label_col].astype(int).to_numpy()
    X_scaled = scaler.transform(X_raw).astype("float32")

    class_weight_dict = get_class_weight_dict(y)

    model.fit(
        X_scaled, y,
        epochs=extra_epochs,
        batch_size=batch_size,
        class_weight=class_weight_dict,
        verbose=0,
    )
    return model


def eval_dense_on_df(
    model: tf.keras.Model,
    df_test: pd.DataFrame,
    scaler: StandardScaler,
    feature_cols,
    label_col: str,
) -> Dict[str, Any]:
    X_raw = df_test[feature_cols].to_numpy().astype("float32")
    y_true = df_test[label_col].astype(int).to_numpy()

    X_scaled = scaler.transform(X_raw).astype("float32")
    proba = model.predict(X_scaled, verbose=0).reshape(-1)
    y_pred = (proba >= 0.5).astype(int)

    return {
        "roc_auc": float(roc_auc_score(y_true, proba)) if len(np.unique(y_true)) > 1 else float("nan"),
        "confusion": confusion_matrix(y_true, y_pred).tolist(),
        "report": classification_report(y_true, y_pred, digits=3, output_dict=True),
        "proba": proba,  # optional: hilfreich für SHAP / Thresholding
    }


# -------------------------
# SVM Pipeline
# -------------------------
def train_svm_pipeline(df_train: pd.DataFrame, feature_cols, label_col: str):
    X = df_train[feature_cols].to_numpy().astype("float32")
    y = df_train[label_col].astype(int).to_numpy()

    svm = make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        SVC(kernel="rbf", probability=False, class_weight="balanced", random_state=42),
    )
    svm.fit(X, y)
    return svm


def eval_svm_on_df(svm_pipeline, df_test: pd.DataFrame, feature_cols, label_col: str) -> Dict[str, Any]:
    X = df_test[feature_cols].to_numpy().astype("float32")
    y_true = df_test[label_col].astype(int).to_numpy()

    # Für ROC-AUC ohne probability=True: decision_function nutzen
    if hasattr(svm_pipeline, "decision_function"):
        score = svm_pipeline.decision_function(X)
        # score kann shape (n,) oder (n,2) sein -> bei binär meist (n,)
        score_1d = score.reshape(-1) if np.ndim(score) == 1 else score[:, -1]
    else:
        score_1d = None

    y_pred = svm_pipeline.predict(X)

    return {
        "roc_auc": float(roc_auc_score(y_true, score_1d)) if (score_1d is not None and len(np.unique(y_true)) > 1) else float("nan"),
        "confusion": confusion_matrix(y_true, y_pred).tolist(),
        "report": classification_report(y_true, y_pred, digits=3, output_dict=True),
        "decision_score": score_1d,  # optional: für SHAP / Ranking / Thresholding
    }


✅ Using: tf.keras


In [8]:
# ============================================================
# (1/3) Compat-Resolver für Fixed Points
# - nutzt _resolve_fixed_points(...) wenn es korrekt (df, points, pairs) zurückgibt
# - ansonsten Fallback über CONFIG / Globals
# ============================================================

from typing import Any, Dict, List, Tuple
import numpy as np
import pandas as pd

def _resolve_fixed_points_fallback(df: pd.DataFrame, label_col: str):
    """
    Fallback, wenn _resolve_fixed_points() None zurückgibt oder fehlt.
    Nutzt (wenn vorhanden):
      CONFIG["fixed_local_ids"] / CONFIG["fixed_local_ilocs"] / CONFIG["id_col"]
    oder globale Variablen:
      FIXED_LOCAL_IDS / FIXED_LOCAL_ILOCS / ID_COL
    """
    cfg = CONFIG if ("CONFIG" in globals() and isinstance(CONFIG, dict)) else {}

    id_col = cfg.get("id_col", None)
    if id_col is None:
        id_col = globals().get("ID_COL", "point_id")
    id_col = id_col or "point_id"

    fixed_ids = cfg.get("fixed_local_ids", None)
    if fixed_ids is None:
        fixed_ids = globals().get("FIXED_LOCAL_IDS", None)

    fixed_ilocs = cfg.get("fixed_local_ilocs", None)
    if fixed_ilocs is None:
        fixed_ilocs = globals().get("FIXED_LOCAL_ILOCS", [398, 1996, 728])
    if fixed_ilocs is None:
        fixed_ilocs = [398, 1996, 728]

    df = df.copy()

    # point_id sicherstellen (erst try: ensure_point_id, sonst selber)
    if "ensure_point_id" in globals() and callable(globals()["ensure_point_id"]):
        df = globals()["ensure_point_id"](df, id_col)
    else:
        if id_col not in df.columns:
            df = df.reset_index(drop=True)
            df[id_col] = np.arange(len(df), dtype=int)
        else:
            df[id_col] = df[id_col].astype(int)

    n = len(df)
    points: List[Dict[str, Any]] = []
    pairs: List[Tuple[int, int]] = []

    # IDs bevorzugen (stabil)
    if fixed_ids is not None:
        id_to_iloc = {int(pid): int(i) for i, pid in enumerate(df[id_col].to_numpy())}
        for pid in fixed_ids:
            pid = int(pid)
            if pid not in id_to_iloc:
                points.append({"point_id": pid, "status": "missing_id"})
                continue
            ix = id_to_iloc[pid]
            y_true = int(df.iloc[ix][label_col]) if label_col in df.columns else None
            points.append({"point_id": pid, "iloc": int(ix), "y_true": y_true, "status": "ok"})
            pairs.append((pid, int(ix)))
        return df, points, pairs

    # sonst ilocs
    for ix in fixed_ilocs:
        ix = int(ix)
        if ix < 0 or ix >= n:
            points.append({"iloc": ix, "status": "invalid_iloc"})
            continue
        pid = int(df.iloc[ix][id_col])
        y_true = int(df.iloc[ix][label_col]) if label_col in df.columns else None
        points.append({"point_id": pid, "iloc": int(ix), "y_true": y_true, "status": "ok"})
        pairs.append((pid, int(ix)))

    return df, points, pairs


def _resolve_fixed_points_compat(df: pd.DataFrame, label_col: str):
    """
    Kompatibel zu:
      - neuer Version: return (df, local_points, pairs)
      - kaputter/alter Version: return None  -> fallback
      - fehlend: fallback
    """
    if "_resolve_fixed_points" in globals() and callable(globals()["_resolve_fixed_points"]):
        res = globals()["_resolve_fixed_points"](df, label_col)
        if isinstance(res, tuple) and len(res) == 3:
            return res
        if res is None:
            return _resolve_fixed_points_fallback(df, label_col)
        return _resolve_fixed_points_fallback(df, label_col)

    return _resolve_fixed_points_fallback(df, label_col)


In [9]:
# ============================================================
# (2/3) Dein run_xai_bundle_for_df — minimal angepasst
# EINZIGE relevante Änderung:
#   _resolve_fixed_points(...)  ->  _resolve_fixed_points_compat(...)
# ============================================================

from typing import Dict, Any
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

def run_xai_bundle_for_df(
    name: str,
    df_source: pd.DataFrame,
    feature_cols,
    label_col: str,
    scaler_A: StandardScaler,
    nn_model,         # dein Keras DenseNN Modell
    svm_pipeline,     # sklearn Pipeline (Imputer->Scaler->SVC)
) -> Dict[str, Any]:
    """
    XAI-Bundle mit festen lokalen Punkten:
      - Anchors (Alibi) lokal auf fixed points (NN + SVM)
    """

    # ------------------------------------------------------------
    # (1) Daten vorbereiten + feste Punkte auflösen
    # ------------------------------------------------------------
    df_source = prepare_df(df_source)

    # >>> FIX: robust/compat (statt direkt _resolve_fixed_points)
    df_source, local_points, pairs = _resolve_fixed_points_compat(df_source, label_col)

    # ------------------------------------------------------------
    # (2) Predict-Funktionen (RAW → Modell) => (n,2) Proba
    # ------------------------------------------------------------
    def nn_pred(X_raw_2d: np.ndarray) -> np.ndarray:
        return nn_predict_proba_raw(nn_model, scaler_A, X_raw_2d)

    # Zusätzlich: Label-Predict für Anchors (Anchors erwartet Klassenlabels, nicht Proba)
    def nn_label_pred(X_raw_2d: np.ndarray) -> np.ndarray:
        proba = nn_pred(np.asarray(X_raw_2d, dtype="float32"))
        return np.argmax(proba, axis=1).astype(int)

    def svm_label_pred(X_raw_2d: np.ndarray) -> np.ndarray:
        X = np.asarray(X_raw_2d, dtype="float32")
        y = svm_pipeline.predict(X)
        return np.asarray(y, dtype=int).reshape(-1)

    # ------------------------------------------------------------
    # (3) Output-Container
    # ------------------------------------------------------------
    out: Dict[str, Any] = {
        "name": name,
        "local_points": local_points,   # inkl. invalid/missing
        "nn": {},
        "svm": {},
            }

    def _safe(tag: str, fn, *args, **kwargs):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            print(f"❌ XAI Fehler bei {tag}: {repr(e)}")
            return {"status": "error", "error": repr(e)}

    seed = int(CONFIG.get("seed", 42)) if "CONFIG" in globals() and isinstance(CONFIG, dict) else 42

    # ------------------------------------------------------------
    # (3b) Anchor-Defaults (optional aus CONFIG ziehen)
    # ------------------------------------------------------------
    anchor_threshold = float(xget("anchor_threshold", 0.90))
    anchor_delta     = float(xget("anchor_delta", 0.10))
    anchor_tau       = float(xget("anchor_tau", 0.05))
    anchor_disc_perc = tuple(xget("anchor_disc_perc", (25, 50, 75)))
    anchor_max_fit_rows = int(xget("anchor_max_fit_rows", 5000))

    # ------------------------------------------------------------
    # (4) DenseNN – XAI
    # ------------------------------------------------------------

    # >>> NEU: Anchors (NN) auf fixed points
    out["nn"]["anchors"] = _safe(
        f"NN Anchors ({name})",
        run_alibi_anchors_fixed_points,
        nn_label_pred,          # Label-Predict!
        df_source,
        feature_cols,
        label_col,
        title=f"NN {name} | Anchors (fixed points)",
        threshold=anchor_threshold,
        delta=anchor_delta,
        tau=anchor_tau,
        disc_perc=anchor_disc_perc,
        max_fit_rows=anchor_max_fit_rows,
        seed=seed,
    )

    # ------------------------------------------------------------
    # (5) SVM – XAI
    # ------------------------------------------------------------

    # >>> NEU: Anchors (SVM) auf fixed points
    out["svm"]["anchors"] = _safe(
        f"SVM Anchors ({name})",
        run_alibi_anchors_fixed_points,
        svm_label_pred,         # Label-Predict!
        df_source,
        feature_cols,
        label_col,
        title=f"SVM {name} | Anchors (fixed points)",
        threshold=anchor_threshold,
        delta=anchor_delta,
        tau=anchor_tau,
        disc_perc=anchor_disc_perc,
        max_fit_rows=anchor_max_fit_rows,
        seed=seed,
    )

    return out


In [10]:
# ============================================================
# 7b) Szenarien (CMAPSS-ähnlich) — übernommen aus patched_v02
#     (damit Keys wie covariance_shift / environmental_shift existieren)
# ============================================================

from typing import Dict, Any, Optional
import numpy as np
import pandas as pd

def get_under_over_train_val(
    df: pd.DataFrame,
    feature_cols,
    label_col: str,
    train_frac: float = 0.7,
    small_train_size: int = 500,
    single_feature: Optional[str] = None,
    min_per_class: int = 1,
) -> Dict[str, Any]:
    """Erzeugt Under-/Overfitting-Splits im CMAPSS-Stil (zeitliche Reihenfolge),
    aber robust gegen das typische Problem: Wenn in Train oder Test nur eine Klasse
    vorkommt, ist ROC-AUC undefiniert und wird NaN.

    Rückgabe:
      {
        "under": (df_train_under, df_test_under, single_feature),
        "over":  (df_train_over,  df_test_over,  single_feature)
      }
    """
    df = df.reset_index(drop=True)
    y = df[label_col].astype(int).to_numpy()
    n = len(df)

    def _has_both_classes(y_arr: np.ndarray) -> bool:
        if y_arr.size == 0:
            return False
        vals, cnts = np.unique(y_arr, return_counts=True)
        if len(vals) < 2:
            return False
        return all(int(c) >= int(min_per_class) for c in cnts)

    def _find_split_near(target_idx: int, max_shift: int) -> int:
        target_idx = int(np.clip(target_idx, 1, n - 1))
        max_shift = int(max(1, min(max_shift, n - 2)))
        for d in range(0, max_shift + 1):
            for cand in (target_idx - d, target_idx + d):
                if cand <= 1 or cand >= n - 1:
                    continue
                if _has_both_classes(y[:cand]) and _has_both_classes(y[cand:]):
                    return int(cand)
        return int(target_idx)

    # 1) Zeitlicher Split, so dass beide Seiten beide Klassen enthalten
    split0 = int(n * float(train_frac))
    split = _find_split_near(split0, max_shift=max(1000, int(0.1 * n)))

    df_train_base = df.iloc[:split].copy()
    df_test_base  = df.iloc[split:].copy()

    # 2) Underfitting: nur 1 Feature variiert (rest konstant aus Train-Statistiken)
    if single_feature is None:
        single_feature = CONFIG.get("under_single_feature", feature_cols[0])

    df_train_under = make_underfitting_view(
        df_train_base, feature_cols, single_feature=single_feature, ref_df=df_train_base
    )
    df_test_under = make_underfitting_view(
        df_test_base, feature_cols, single_feature=single_feature, ref_df=df_train_base
    )

    # 3) Overfitting: sehr kleines Trainset, aber falls nötig erweitern bis beide Klassen drin sind
    train_end = int(min(max(2, small_train_size), len(df_train_base)))
    while train_end < len(df_train_base) and not _has_both_classes(y[:train_end]):
        train_end += 1
    while train_end < n - 1 and not _has_both_classes(y[:train_end]):
        train_end += 1

    df_train_over = df.iloc[:train_end].copy()
    df_test_over  = df.iloc[train_end:].copy()

    return {
        "under": (df_train_under, df_test_under, single_feature),
        "over":  (df_train_over,  df_test_over,  single_feature),
    }

def make_scenarios_like_cmapss(df_train_A: pd.DataFrame, df_test_A: pd.DataFrame, feature_cols) -> Dict[str, Dict[str, Any]]:
    scenarios = {"baseline": {"train": df_train_A, "test": df_test_A, "meta": {}}}

    # ---------------------------
    # Covariance shift (eval-only)
    # ---------------------------
    df_test_cov, meta_cov = apply_covariance_shift_df(
        df_test_A, feature_cols,
        scale=CONFIG["cov_shift_scale"],
        frac_features=CONFIG["cov_shift_frac_features"],
        seed=CONFIG["seed"] + 10
    )
    scenarios["covariance_shift"] = {"train": df_train_A, "test": df_test_cov, "meta": meta_cov}

    # ---------------------------
    # Environmental shift (eval-only)
    # ---------------------------
    df_test_env, meta_env = apply_environmental_shift_df(
        df_test_A, feature_cols,
        bias=CONFIG["env_shift_bias"],
        frac_features=CONFIG["env_shift_frac_features"],
        seed=CONFIG["seed"] + 20
    )
    scenarios["environmental_shift"] = {"train": df_train_A, "test": df_test_env, "meta": meta_env}

    # ---------------------------
    # Model drift (MODEL-only: drifted TRAIN, baseline TEST)
    # ---------------------------
    df_train_drift, meta_cov_tr = apply_covariance_shift_df(
        df_train_A, feature_cols,
        scale=CONFIG["cov_shift_scale"],
        frac_features=CONFIG["cov_shift_frac_features"],
        seed=CONFIG["seed"] + 30
    )
    df_train_drift, meta_env_tr = apply_environmental_shift_df(
        df_train_drift, feature_cols,
        bias=CONFIG["env_shift_bias"],
        frac_features=CONFIG["env_shift_frac_features"],
        seed=CONFIG["seed"] + 31
    )

    # TEST bleibt bewusst unverändert -> wir testen das *Modell* (drifted weights), nicht das Dataset
    scenarios["model_drift"] = {
        "train": df_train_drift,
        "test":  df_test_A,
        "meta": {
            "train": {"cov": meta_cov_tr, "env": meta_env_tr},
            "test":  {"note": "baseline test (no data drift)"},
        },
    }

    # ---------------------------
    # Catastrophic forgetting (Domain B)
    # ---------------------------
    df_train_B, meta_B_tr = make_domain_B_from_A(df_train_A, feature_cols, seed=CONFIG["seed"] + 40)
    df_test_B,  meta_B_te = make_domain_B_from_A(df_test_A,  feature_cols, seed=CONFIG["seed"] + 41)

    scenarios["catastrophic_forgetting"] = {
        "train_A": df_train_A, "test_A": df_test_A,
        "train_B": df_train_B, "test_B": df_test_B,
        "meta": {"B_train": meta_B_tr, "B_test": meta_B_te},
    }

    # ============================================================
    # Underfitting / Overfitting → MODEL-ONLY
    # ============================================================
    df_full_A = pd.concat([df_train_A, df_test_A], axis=0).reset_index(drop=True)

    splits = get_under_over_train_val(
        df=df_full_A,
        feature_cols=feature_cols,
        label_col=label_col,
        train_frac=CONFIG.get("train_frac", 0.7),
        small_train_size=CONFIG.get("over_small_train_size", 500),
        single_feature=CONFIG.get("under_single_feature", None),
    )

    df_train_under, _, sf = splits["under"]
    df_train_over,  _, _  = splits["over"]

    # TESTSET BLEIBT BASELINE!
    scenarios["underfitting"] = {
        "train": df_train_under,
        "test":  df_test_A,
        "meta": {"model_only": True, "single_feature": sf},
    }

    scenarios["overfitting"] = {
        "train": df_train_over,
        "test":  df_test_A,
        "meta": {"model_only": True, "small_train_size": len(df_train_over)},
    }


    return scenarios


In [11]:
# ============================================================
# 8) MAIN (angepasst: überall konsistente DF-Preprocessing + XAI(Anchors))
# ============================================================

# ============================================================
# Missing helpers (aus patched_v02 übernommen)
# -> behebt: NameError: make_underfitting_view is not defined
# + außerdem: nn_predict_proba_raw (für Anchors/NN label pred robust)
# ============================================================

# ============================================================
# Alibi Anchors (AnchorTabular) — FIXED POINTS — NN + SVM
# ============================================================

from typing import Any, Dict, Optional, Sequence, Callable
import numpy as np
import pandas as pd
from alibi.explainers import AnchorTabular

def _subsample_fit_X(X: np.ndarray, max_n: int, seed: int) -> np.ndarray:
    if max_n is None or len(X) <= int(max_n):
        return X
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(X), size=int(max_n), replace=False)
    return X[idx]

def run_alibi_anchors_fixed_points(
    predict_labels_fn: Callable[[np.ndarray], np.ndarray],
    df_source: pd.DataFrame,
    feature_cols: Sequence[str],
    label_col: str,
    title: str = "Anchors",
    *,
    threshold: float = 0.90,
    delta: float = 0.10,
    tau: float = 0.05,
    disc_perc: tuple = (25, 50, 75),
    max_fit_rows: int = 5000,
    seed: int = 42,
    categorical_names: Optional[Dict[int, Sequence[str]]] = None,
) -> Dict[str, Any]:

    print(f"\n\n############################")
    print(f"ALIBI ANCHORS (fixed points) für: {title}")
    print(f"############################\n")

    # Muss existieren:
    # df_source, local_points, pairs = _resolve_fixed_points(df_source, label_col)
    # (oder _resolve_fixed_points_compat, je nach deinem Notebook)
    df_source, local_points, pairs = _resolve_fixed_points_compat(df_source, label_col)

    if not pairs:
        print("Keine gültigen Fixed Points gefunden.")
        return {"status": "skipped", "reason": "no valid fixed points", "local_points": local_points, "pairs": []}

    X_raw_all = df_source[list(feature_cols)].to_numpy().astype("float32")
    y_true_all = df_source[label_col].astype(int).to_numpy() if label_col in df_source.columns else None

    if categorical_names is None:
        categorical_names = {}

    X_fit = _subsample_fit_X(X_raw_all, max_fit_rows, seed=seed)

    # alibi API ist je nach Version unterschiedlich -> robust bauen
    try:
    # häufige Signatur: AnchorTabular(predictor, feature_names, categorical_names=...)
        explainer = AnchorTabular(predict_labels_fn, list(feature_cols), categorical_names=categorical_names)
    except TypeError:
        # alternative Signatur (manche Builds akzeptieren keywords)
        explainer = AnchorTabular(predictor=predict_labels_fn, feature_names=list(feature_cols), categorical_names=categorical_names)

    explainer.fit(X_fit, disc_perc=disc_perc)

    out: Dict[str, Any] = {
        "status": "ok",
        "title": title,
        "meta": {
            "threshold": float(threshold),
            "delta": float(delta),
            "tau": float(tau),
            "disc_perc": disc_perc,
            "fit_rows_used": int(len(X_fit)),
            "n_total_rows": int(len(X_raw_all)),
        },
        "local_points": local_points,
        "pairs": pairs,
        "per_point": {},
    }

    for pid, ix in pairs:
        x_raw = X_raw_all[ix:ix+1]
        y_pred = int(np.asarray(predict_labels_fn(x_raw)).reshape(-1)[0])
        y_true = int(y_true_all[ix]) if y_true_all is not None else None

        exp = explainer.explain(x_raw, threshold=threshold, delta=delta, tau=tau)

        rules = list(getattr(exp, "anchor", [])) if getattr(exp, "anchor", None) else []
        precision = float(getattr(exp, "precision", np.nan))
        coverage = float(getattr(exp, "coverage", np.nan))

        out["per_point"][int(pid)] = {
            "point_id": int(pid),
            "iloc": int(ix),
            "y_true": y_true,
            "y_pred": y_pred,
            "anchor_rules": rules,
            "precision": precision,
            "coverage": coverage,
        }

    return out


import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# 1) Underfitting-View (aus patched_v02)
if "make_underfitting_view" not in globals():
    def make_underfitting_view(df: pd.DataFrame, feature_cols, single_feature: str, ref_df: pd.DataFrame) -> pd.DataFrame:
        """Alle Features bleiben, aber nur single_feature variiert; Rest konstant (Mittelwert aus ref_df)."""
        df_u = df.copy()
        means = ref_df[feature_cols].mean(numeric_only=True)
        for c in feature_cols:
            if c != single_feature:
                df_u[c] = means[c]
        return df_u


# 2) NN proba helper (aus patched_v02) – damit Anchors sauber Labels ableiten können
if "nn_predict_proba_raw" not in globals():
    def nn_predict_proba_raw(model, scaler: StandardScaler, X_raw_2d: np.ndarray) -> np.ndarray:
        """
        DenseNN: erwartet i.d.R. (n, f). Falls das Modell 3D erwartet,
        reshapen wir automatisch.
        Rückgabe: (n, 2) mit [P(class0), P(class1)]
        """
        X_scaled = scaler.transform(X_raw_2d.astype("float32")).astype("float32")

        # Auto-reshape falls Modell 3D erwartet (z.B. CNN)
        in_shape = getattr(model, "input_shape", None)
        X_in = X_scaled
        if in_shape is not None and len(in_shape) == 3:
            X_in = X_scaled.reshape(-1, X_scaled.shape[1], 1)

        # Keras predict kompatibel
        try:
            y = model.predict(X_in, verbose=0)
        except TypeError:
            y = model.predict(X_in)

        y = np.asarray(y).reshape(-1)

        # Binary output -> (n,2)
        if y.ndim == 1:
            p1 = y.astype("float32")
            p0 = (1.0 - p1).astype("float32")
            return np.vstack([p0, p1]).T

        # Falls Model schon (n,2) liefert:
        if y.ndim == 2 and y.shape[1] == 2:
            return y.astype("float32")

        # Fallback: argmax -> onehot
        y2 = np.asarray(model.predict(X_in), dtype="float32")
        cls = np.argmax(y2, axis=1).astype(int)
        p1 = cls.astype("float32")
        return np.vstack([1.0 - p1, p1]).T


# 3) xget fallback (nur falls im Notebook nicht vorhanden)
if "xget" not in globals():
    def _xai_cfg() -> dict:
        if "CONFIG" in globals() and isinstance(CONFIG, dict):
            return CONFIG.get("xai", CONFIG)
        return {}
    def xget(key: str, default):
        return _xai_cfg().get(key, default)


df = prepare_df(df_base)

# Stable IDs for comparing the same logical point across scenarios
ID_COL = CONFIG.get("id_col", "point_id")
df = ensure_point_id(df, ID_COL)

# Domain A: zeitbasiert splitten
df_train_A, df_test_A = split_time_ordered(df, CONFIG["train_frac"])
df_tr_A, df_val_A = make_train_val(df_train_A, CONFIG["val_frac_within_train"])

# Baseline-Scaler nur aus Train (Domain A)
scaler_A = make_scaler_from_train(df_tr_A, feature_cols)

# ------------------------------------------------------------
# Train clean models (Domain A)
# ------------------------------------------------------------
nn_clean = train_dense_on_df(
    df_train=df_tr_A,
    scaler=scaler_A,
    feature_cols=feature_cols,
    label_col=label_col,
    epochs=CONFIG.get("nn_epochs", 200),
    batch_size=CONFIG.get("nn_batch_size", 32),
    seed=CONFIG["seed"],
)

svm_clean = train_svm_pipeline(df_tr_A, feature_cols, label_col)

# Szenarien (CMAPSS-ähnlich)
scenarios = make_scenarios_like_cmapss(df_train_A=df_tr_A, df_test_A=df_test_A, feature_cols=feature_cols)
ALL_RESULTS = {}

# ------------------------------------------------------------
# Helper: DF hart auf feature_cols ausrichten (für alle Szenarien!)
# ------------------------------------------------------------
def align_df_to_features(df: pd.DataFrame, feature_cols, fill_value=0.0) -> pd.DataFrame:
    df = df.copy()
    missing = [c for c in feature_cols if c not in df.columns]
    for c in missing:
        df[c] = fill_value
    return df

# ------------------------------------------------------------
# Helper: einheitliches Preprocessing für ALLE Eval-/XAI-DFs
# (verhindert u.a. fehlendes 'name' / fehlende point_id / fehlende Features)
# ------------------------------------------------------------
def prep_eval_df(df_in: pd.DataFrame) -> pd.DataFrame:
    d = prepare_df(df_in)
    d = ensure_point_id(d, ID_COL)
    d = align_df_to_features(d, feature_cols, fill_value=0.0)
    return d


# ------------------------------------------------------------
# 8.1 baseline / cov / env: evaluate-only shifts (no retraining)
# ------------------------------------------------------------
for name in ["baseline", "covariance_shift", "environmental_shift"]:
    sc = scenarios[name]

    # <- WICHTIG: hier ist 'name' definiert (kein NameError)
    df_test = prep_eval_df(sc["test"])

    met_nn  = eval_dense_on_df(nn_clean,  df_test, scaler_A, feature_cols, label_col)
    met_svm = eval_svm_on_df(svm_clean,   df_test, feature_cols, label_col)

    xai = run_xai_bundle_for_df(
        name=name,
        df_source=df_test,
        feature_cols=feature_cols,
        label_col=label_col,
        scaler_A=scaler_A,
        nn_model=nn_clean,
        svm_pipeline=svm_clean,
    )

    ALL_RESULTS[name] = {
        "metrics": {"nn": met_nn, "svm": met_svm},
        "xai": xai,
        "meta": sc.get("meta", {}),
    }


# ------------------------------------------------------------
# Helper: "clone model" (tf.keras oder standalone keras)
# ------------------------------------------------------------
def _clone_keras_model(m):
    try:
        import tensorflow as tf
        if hasattr(tf, "keras"):
            cm = tf.keras.models.clone_model(m)
            cm.set_weights(m.get_weights())
            return cm
    except Exception:
        pass

    import keras
    cm = keras.models.clone_model(m)
    cm.set_weights(m.get_weights())
    return cm

def _compile_dense_like_original(m):
    m.compile(
        optimizer=optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=[metrics.AUC(name="auc"), metrics.Precision(name="prec"), metrics.Recall(name="rec")],
    )
    return m


# ------------------------------------------------------------
# 8.2 model drift: MODEL-only (Train drifted, Test bleibt clean)
# ------------------------------------------------------------
sc = scenarios["model_drift"]

nn_drift = _clone_keras_model(nn_clean)
nn_drift = _compile_dense_like_original(nn_drift)

nn_drift = continue_train_dense(
    nn_drift,
    sc["train"],
    scaler_A,
    feature_cols,
    label_col,
    extra_epochs=CONFIG.get("drift_extra_epochs", 20),
    batch_size=CONFIG.get("nn_batch_size", 32),
)

svm_drift = (
    train_svm_pipeline(sc["train"], feature_cols, label_col)
    if CONFIG.get("svm_retrain_on_drift", True)
    else svm_clean
)

df_test = prep_eval_df(sc["test"])

met_nn  = eval_dense_on_df(nn_drift,  df_test, scaler_A, feature_cols, label_col)
met_svm = eval_svm_on_df(svm_drift,   df_test, feature_cols, label_col)

xai = run_xai_bundle_for_df(
    name="model_drift",
    df_source=df_test,
    feature_cols=feature_cols,
    label_col=label_col,
    scaler_A=scaler_A,
    nn_model=nn_drift,
    svm_pipeline=svm_drift,
)

ALL_RESULTS["model_drift"] = {
    "metrics": {"nn": met_nn, "svm": met_svm},
    "xai": xai,
    "meta": sc.get("meta", {}),
}


# ------------------------------------------------------------
# 8.3 catastrophic forgetting: A -> train on B -> eval on A and B
# ------------------------------------------------------------
sc = scenarios["catastrophic_forgetting"]
nn_A = nn_clean
svm_A = svm_clean

nn_B = _clone_keras_model(nn_clean)
nn_B = _compile_dense_like_original(nn_B)
nn_B = continue_train_dense(
    nn_B,
    sc["train_B"],
    scaler_A,
    feature_cols,
    label_col,
    extra_epochs=CONFIG.get("forget_ft_epochs", 20),
    batch_size=CONFIG.get("nn_batch_size", 32),
)

svm_B = (
    train_svm_pipeline(sc["train_B"], feature_cols, label_col)
    if CONFIG.get("forget_svm_retrain", True)
    else svm_A
)

df_test_A_eval = prep_eval_df(sc["test_A"])
df_test_B_eval = prep_eval_df(sc["test_B"])

met_AA = {
    "nn":  eval_dense_on_df(nn_A,  df_test_A_eval, scaler_A, feature_cols, label_col),
    "svm": eval_svm_on_df(svm_A,   df_test_A_eval, feature_cols, label_col),
}
met_BA = {
    "nn":  eval_dense_on_df(nn_B,  df_test_A_eval, scaler_A, feature_cols, label_col),
    "svm": eval_svm_on_df(svm_B,   df_test_A_eval, feature_cols, label_col),
}
met_BB = {
    "nn":  eval_dense_on_df(nn_B,  df_test_B_eval, scaler_A, feature_cols, label_col),
    "svm": eval_svm_on_df(svm_B,   df_test_B_eval, feature_cols, label_col),
}

xai_AA = run_xai_bundle_for_df(
    name="forgetting_A(onA)",
    df_source=df_test_A_eval,
    feature_cols=feature_cols,
    label_col=label_col,
    scaler_A=scaler_A,
    nn_model=nn_A,
    svm_pipeline=svm_A,
)
xai_BA = run_xai_bundle_for_df(
    name="forgetting_B(onA)",
    df_source=df_test_A_eval,
    feature_cols=feature_cols,
    label_col=label_col,
    scaler_A=scaler_A,
    nn_model=nn_B,
    svm_pipeline=svm_B,
)
xai_BB = run_xai_bundle_for_df(
    name="forgetting_B(onB)",
    df_source=df_test_B_eval,
    feature_cols=feature_cols,
    label_col=label_col,
    scaler_A=scaler_A,
    nn_model=nn_B,
    svm_pipeline=svm_B,
)

ALL_RESULTS["catastrophic_forgetting"] = {
    "metrics": {"A_on_A": met_AA, "B_on_A": met_BA, "B_on_B": met_BB},
    "xai": {"A_on_A": xai_AA, "B_on_A": xai_BA, "B_on_B": xai_BB},
    "meta": sc.get("meta", {}),
}


# ------------------------------------------------------------
# 8.4 underfitting / overfitting: eigenes Training pro Szenario
# ------------------------------------------------------------
for name in ["underfitting", "overfitting"]:
    sc = scenarios[name]

    scaler_local = make_scaler_from_train(sc["train"], feature_cols)

    if name == "underfitting":
        epochs = CONFIG.get("under_epochs", CONFIG.get("underover_epochs", 120))
    else:
        epochs = CONFIG.get("over_epochs", CONFIG.get("underover_epochs", 120))

    nn_local = train_dense_on_df(
        df_train=sc["train"],
        scaler=scaler_local,
        feature_cols=feature_cols,
        label_col=label_col,
        epochs=epochs,
        batch_size=CONFIG.get("nn_batch_size", 32),
        seed=CONFIG["seed"],
    )

    svm_local = train_svm_pipeline(sc["train"], feature_cols, label_col)

    df_test = prep_eval_df(sc["test"])

    met_nn  = eval_dense_on_df(nn_local,  df_test, scaler_local, feature_cols, label_col)
    met_svm = eval_svm_on_df(svm_local,   df_test, feature_cols, label_col)

    xai = run_xai_bundle_for_df(
        name=name,
        df_source=df_test,
        feature_cols=feature_cols,
        label_col=label_col,
        scaler_A=scaler_local,
        nn_model=nn_local,
        svm_pipeline=svm_local,
    )

    ALL_RESULTS[name] = {
        "metrics": {"nn": met_nn, "svm": met_svm},
        "xai": xai,
        "meta": sc.get("meta", {}),
    }

print("DONE. Scenarios:", list(ALL_RESULTS.keys()))


Instructions for updating:
Colocations handled automatically by placer.


C:\Users\ko\Anaconda3\envs\Diss_Alibi\lib\site-packages\keras\engine\training_v1.py:2332: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  updates = self.state_updates
C:\Users\ko\Anaconda3\envs\Diss_Alibi\lib\site-packages\keras\engine\training_v1.py:2356: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  updates=self.state_updates,




############################
ALIBI ANCHORS (fixed points) für: NN baseline | Anchors (fixed points)
############################



Could not find an anchor satisfying the 0.9 precision constraint. Now returning the best non-eligible result. The desired precision threshold might not be achieved due to the quantile-based discretisation of the numerical features. The resolution of the bins may be too large to find an anchor of required precision. Consider increasing the number of bins in `disc_perc`, but note that for some numerical distribution (e.g. skewed distribution) it may not help.




############################
ALIBI ANCHORS (fixed points) für: SVM baseline | Anchors (fixed points)
############################



...

C:\Temp\ipykernel_24920\3729372851.py:98: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  precision = float(getattr(exp, "precision", np.nan))
C:\Temp\ipykernel_24920\3729372851.py:98: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  precision = float(getattr(exp, "precision", np.nan))
C:\Temp\ipykernel_24920\3729372851.py:98: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  precision = float(getattr(exp, "precision", np.nan))
C:\Users\ko\Anaconda3\envs\Diss_Alibi\lib\site



############################
ALIBI ANCHORS (fixed points) für: NN overfitting | Anchors (fixed points)
############################



Could not find an anchor satisfying the 0.9 precision constraint. Now returning the best non-eligible result. The desired precision threshold might not be achieved due to the quantile-based discretisation of the numerical features. The resolution of the bins may be too large to find an anchor of required precision. Consider increasing the number of bins in `disc_perc`, but note that for some numerical distribution (e.g. skewed distribution) it may not help.
Could not find an anchor satisfying the 0.9 precision constraint. Now returning the best non-eligible result. The desired precision threshold might not be achieved due to the quantile-based discretisation of the numerical features. The resolution of the bins may be too large to find an anchor of required precision. Consider increasing the number of bins in `disc_perc`, but note that for some numerical distribution (e.g. skewed distribution) it may not help.




############################
ALIBI ANCHORS (fixed points) für: SVM overfitting | Anchors (fixed points)
############################

DONE. Scenarios: ['baseline', 'covariance_shift', 'environmental_shift', 'model_drift', 'catastrophic_forgetting', 'underfitting', 'overfitting']


C:\Temp\ipykernel_24920\3729372851.py:98: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  precision = float(getattr(exp, "precision", np.nan))
C:\Temp\ipykernel_24920\3729372851.py:98: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  precision = float(getattr(exp, "precision", np.nan))
C:\Temp\ipykernel_24920\3729372851.py:98: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  precision = float(getattr(exp, "precision", np.nan))


In [12]:
# ============================================================
# 9) Übersicht: XAI pro Szenario & Modell (JETZT: nur Anchors für NN + SVM)
# ============================================================
from typing import Dict, Any
import numpy as np

def print_xai_overview(ALL_RESULTS: Dict[str, Any]):
    print("=" * 90)
    print("XAI OVERVIEW pro Szenario / Modell (Anchors: NN + SVM)")
    print("=" * 90)

    def _status(x):
        if isinstance(x, dict):
            return x.get("status", "n/a")
        return "n/a"

    def _has_error(x):
        return isinstance(x, dict) and x.get("status") == "error"

    def _anchors_summary(a: Dict[str, Any]) -> Dict[str, Any]:
        """
        Liefert: explained_points, mean_precision, mean_coverage
        """
        if not isinstance(a, dict) or a.get("status") != "ok":
            return {"explained_points": 0, "mean_precision": None, "mean_coverage": None}

        per_point = a.get("per_point", {}) if isinstance(a.get("per_point", {}), dict) else {}
        n = len(per_point)

        precisions = []
        coverages = []
        for _, rec in per_point.items():
            if isinstance(rec, dict):
                p = rec.get("precision", None)
                c = rec.get("coverage", None)
                if isinstance(p, (int, float)) and not np.isnan(p):
                    precisions.append(float(p))
                if isinstance(c, (int, float)) and not np.isnan(c):
                    coverages.append(float(c))

        mean_p = float(np.mean(precisions)) if precisions else None
        mean_c = float(np.mean(coverages)) if coverages else None

        return {"explained_points": n, "mean_precision": mean_p, "mean_coverage": mean_c}

    def one_bundle(name: str, bundle: Dict[str, Any]):
        print("\n" + "-" * 90)
        print("SCENARIO:", name)

        nn  = bundle.get("nn", {}) if isinstance(bundle, dict) else {}
        svm = bundle.get("svm", {}) if isinstance(bundle, dict) else {}

        # --- NN Anchors ---
        if isinstance(nn, dict) and nn:
            a = nn.get("anchors", {})
            summ = _anchors_summary(a)
            print("[NN ] Anchors (Alibi)")
            print("  status        :", _status(a))
            print("  points        :", summ["explained_points"])
            print("  mean_precision:", summ["mean_precision"])
            print("  mean_coverage :", summ["mean_coverage"])
            if _has_error(a):
                print("  ⚠️  NN Anchors: Fehler (siehe payload).")

        # --- SVM Anchors ---
        if isinstance(svm, dict) and svm:
            a = svm.get("anchors", {})
            summ = _anchors_summary(a)
            print("[SVM] Anchors (Alibi)")
            print("  status        :", _status(a))
            print("  points        :", summ["explained_points"])
            print("  mean_precision:", summ["mean_precision"])
            print("  mean_coverage :", summ["mean_coverage"])
            if _has_error(a):
                print("  ⚠️  SVM Anchors: Fehler (siehe payload).")

        # --- Fixed points summary (optional)
        lp = bundle.get("local_points", None) if isinstance(bundle, dict) else None
        if isinstance(lp, list):
            ok = sum(1 for p in lp if isinstance(p, dict) and p.get("status") == "ok")
            bad = len(lp) - ok
            print(f"[FixedLocalPoints] total={len(lp)} ok={ok} other={bad}")

    for scen, payload in (ALL_RESULTS or {}).items():
        if scen != "catastrophic_forgetting":
            # erwartet: ALL_RESULTS[scen]["xai"] enthält bundle
            bundle = (payload.get("xai", {}) if isinstance(payload, dict) else {})
            one_bundle(scen, bundle)
        else:
            # forgetting payload contains xai dict with subkeys
            xai = payload.get("xai", {}) if isinstance(payload, dict) else {}
            for subk in ["A_on_A", "B_on_A", "B_on_B"]:
                one_bundle(f"{scen}::{subk}", xai.get(subk, {}))

# Aufruf:
print_xai_overview(ALL_RESULTS)


XAI OVERVIEW pro Szenario / Modell (Anchors: NN + SVM)

------------------------------------------------------------------------------------------
SCENARIO: baseline
[NN ] Anchors (Alibi)
  status        : ok
  points        : 3
  mean_precision: 0.9032942646600408
  mean_coverage : 0.2514
[SVM] Anchors (Alibi)
  status        : ok
  points        : 3
  mean_precision: 0.7640214287074397
  mean_coverage : 0.2507
[FixedLocalPoints] total=3 ok=3 other=0

------------------------------------------------------------------------------------------
SCENARIO: covariance_shift
[NN ] Anchors (Alibi)
  status        : ok
  points        : 3
  mean_precision: 1.0
  mean_coverage : 1.0
[SVM] Anchors (Alibi)
  status        : ok
  points        : 3
  mean_precision: 1.0
  mean_coverage : 1.0
[FixedLocalPoints] total=3 ok=3 other=0

------------------------------------------------------------------------------------------
SCENARIO: environmental_shift
[NN ] Anchors (Alibi)
  status        : ok
  poin

In [13]:
import os
from pprint import pformat

def _safe_str(obj, max_chars=12000):
    try:
        s = pformat(obj, width=120)
    except Exception:
        try:
            s = str(obj)
        except Exception:
            s = repr(obj)
    if len(s) > max_chars:
        s = s[:max_chars] + "\n... [TRUNCATED] ..."
    return s


def _iter_scenarios_with_xai(all_results: dict):
    """
    Yields tuples: (scenario_label, payload_like)
    Berücksichtigt catastrophic_forgetting Subsets.
    """
    for scen, payload in all_results.items():
        if scen != "catastrophic_forgetting":
            yield scen, payload
        else:
            xai = payload.get("xai", {}) or {}
            met = payload.get("metrics", {}) or {}
            meta = payload.get("meta", None)
            for subk in ["A_on_A", "B_on_A", "B_on_B"]:
                yield f"{scen}::{subk}", {
                    "xai": xai.get(subk, {}),
                    "metrics": met.get(subk, {}),
                    "meta": meta,
                }


def export_xai_results_to_txt(all_results: dict, out_path: str = "XAI_RESULTS_NN_SVM.txt"):
    """
    Schreibt eine lesbare TXT:
    SCENARIO -> MODEL (nn/svm/proto_crit) -> METHOD -> point_id (falls vorhanden)
    """
    lines = []

    for scen_label, payload in _iter_scenarios_with_xai(all_results):
        lines.append("=" * 90)
        lines.append(f"SCENARIO: {scen_label}")
        lines.append("=" * 90)

        xai = payload.get("xai", None)
        if not isinstance(xai, dict):
            lines.append("[WARN] No xai dict found.\n")
            continue

        # Neue Struktur: nn/svm/proto_crit + optional local_points
        model_blocks = []
        for k in ["nn", "svm", "proto_crit"]:
            if k in xai:
                model_blocks.append((k, xai.get(k)))
        # Fallback: falls anders aufgebaut
        if not model_blocks:
            model_blocks = [("xai", xai)]

        # FixedLocalPoints Overview
        lp = xai.get("local_points", None)
        if isinstance(lp, list):
            ok = sum(1 for p in lp if isinstance(p, dict) and p.get("status") == "ok")
            lines.append(f"\n[FixedLocalPoints] total={len(lp)} ok={ok} other={len(lp)-ok}")

        for model_name, model_xai in model_blocks:
            lines.append(f"\n--- MODEL: {model_name} ---")

            if not isinstance(model_xai, dict):
                lines.append(_safe_str(model_xai))
                continue

            # Methoden sortiert ausgeben
            for method_name in sorted(model_xai.keys()):
                lines.append(f"\n[METHOD] {method_name}")
                val = model_xai[method_name]

                # 1) SHAP local Struktur: {"per_point": {pid: {...}}}
                if isinstance(val, dict) and "per_point" in val and isinstance(val["per_point"], dict):
                    per_point = val["per_point"]
                    lines.append(f"  status: {val.get('status', 'n/a')}")
                    for pid in sorted(per_point.keys()):
                        lines.append(f"\n  point_id={pid}")
                        lines.append(_safe_str(per_point[pid]))
                    continue

                # 2) dict keyed by int point_id (selten bei unserem neuen Code, aber unterstützend)
                if isinstance(val, dict) and val and all(isinstance(k, int) for k in val.keys()):
                    for pid in sorted(val.keys()):
                        lines.append(f"\n  point_id={pid}")
                        lines.append(_safe_str(val[pid]))
                    continue

                # 3) normaler dump
                lines.append(_safe_str(val))

        # Metrics optional (kurz)
        met = payload.get("metrics", None)
        if met is not None:
            lines.append("\n--- METRICS (raw) ---")
            lines.append(_safe_str(met))

        # Meta optional
        meta = payload.get("meta", None)
        if meta is not None:
            lines.append("\n--- META ---")
            lines.append(_safe_str(meta))

        lines.append("\n")

    with open(out_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

    print(f"Wrote: {os.path.abspath(out_path)}")


# Beispiel:
export_xai_results_to_txt(ALL_RESULTS, "XAI_RESULTS_NN_SVM.txt")


Wrote: C:\Users\ko\XAI_RESULTS_NN_SVM.txt


In [14]:
def print_local_xai_point_ids(ALL_RESULTS):
    for scen, payload in ALL_RESULTS.items():

        # catastrophic_forgetting hat Subsets
        if scen == "catastrophic_forgetting":
            for subk in ["A_on_A", "B_on_A", "B_on_B"]:
                xai = (payload.get("xai", {}) or {}).get(subk, {})
                print("\n====", f"{scen}::{subk}", "====")
                _print_one_xai_block(xai)
            continue

        xai = payload.get("xai", {})
        print("\n====", scen, "====")
        _print_one_xai_block(xai)


def _print_one_xai_block(xai: dict):
    # local points
    lp = xai.get("local_points", [])
    print("Local points:", lp)

    # NN
    if "nn" in xai:
        nn = xai["nn"]

        # Anchors (per_point: pid -> details)
        a = nn.get("anchors", {})
        per = a.get("per_point", {}) if isinstance(a, dict) else {}
        print("NN Anchors point_ids:", list(per.keys()))

    # SVM
    if "svm" in xai:
        svm = xai["svm"]

        # Anchors (per_point: pid -> details)
        a = svm.get("anchors", {})
        per = a.get("per_point", {}) if isinstance(a, dict) else {}
        print("SVM Anchors point_ids:", list(per.keys()))


# Aufruf
print_local_xai_point_ids(ALL_RESULTS)



==== baseline ====
Local points: [{'point_id': 55838, 'iloc': 398, 'y_true': 0, 'status': 'ok'}, {'point_id': 57436, 'iloc': 1996, 'y_true': 1, 'status': 'ok'}, {'point_id': 56168, 'iloc': 728, 'y_true': 1, 'status': 'ok'}]
NN Anchors point_ids: [55838, 57436, 56168]
SVM Anchors point_ids: [55838, 57436, 56168]

==== covariance_shift ====
Local points: [{'point_id': 55838, 'iloc': 398, 'y_true': 0, 'status': 'ok'}, {'point_id': 57436, 'iloc': 1996, 'y_true': 1, 'status': 'ok'}, {'point_id': 56168, 'iloc': 728, 'y_true': 1, 'status': 'ok'}]
NN Anchors point_ids: [55838, 57436, 56168]
SVM Anchors point_ids: [55838, 57436, 56168]

==== environmental_shift ====
Local points: [{'point_id': 55838, 'iloc': 398, 'y_true': 0, 'status': 'ok'}, {'point_id': 57436, 'iloc': 1996, 'y_true': 1, 'status': 'ok'}, {'point_id': 56168, 'iloc': 728, 'y_true': 1, 'status': 'ok'}]
NN Anchors point_ids: [55838, 57436, 56168]
SVM Anchors point_ids: [55838, 57436, 56168]

==== model_drift ====
Local points: [{

In [15]:
a = xai["nn"]["anchors"]
print("status:", a.get("status"))
print("pairs:", a.get("pairs"))
print("local_points:", a.get("local_points"))

status: ok
pairs: [(55838, 398), (57436, 1996), (56168, 728)]
local_points: [{'point_id': 55838, 'iloc': 398, 'y_true': 0, 'status': 'ok'}, {'point_id': 57436, 'iloc': 1996, 'y_true': 1, 'status': 'ok'}, {'point_id': 56168, 'iloc': 728, 'y_true': 1, 'status': 'ok'}]


In [16]:
def print_anchors(xai, which="nn", max_points=10):
    a = xai[which]["anchors"]
    print(which.upper(), "status:", a.get("status"))
    per = a.get("per_point", {}) or {}
    print("points:", len(per))
    for i, (pid, rec) in enumerate(per.items()):
        if i >= max_points:
            break
        print(f"\n[{which}] point_id={pid} iloc={rec.get('iloc')}")
        print("precision:", rec.get("precision"), "coverage:", rec.get("coverage"))
        print("rules:")
        for r in (rec.get("anchor_rules") or []):
            print(" -", r)

print_anchors(xai, "nn")
print_anchors(xai, "svm")


NN status: ok
points: 3

[nn] point_id=55838 iloc=398
precision: 0.37147425647585547 coverage: 0.245
rules:
 - mittelwert <= 341.11

[nn] point_id=57436 iloc=1996
precision: 0.37196039603960396 coverage: 0.2509
rules:
 - mittelwert <= 341.11

[nn] point_id=56168 iloc=728
precision: 1.0 coverage: 0.2448
rules:
 - mittelwert > 363.44
SVM status: ok
points: 3

[svm] point_id=55838 iloc=398
precision: 0.98 coverage: 1.0
rules:

[svm] point_id=57436 iloc=1996
precision: 1.0 coverage: 1.0
rules:

[svm] point_id=56168 iloc=728
precision: 1.0 coverage: 1.0
rules:


In [17]:
from typing import Optional

def print_anchor_rules_for_error_scenarios(
    ALL_RESULTS,
    *,
    include_baseline: bool = False,
    only_misclassified: bool = False,
    max_points: Optional[int] = None,
):
    """
    Gibt für jedes (Fehler-)Szenario die Anchor-Regeln pro Fixed-Point aus.
    - include_baseline=False: baseline wird ausgelassen (typisch kein "Fehlerszenario")
    - only_misclassified=True: nur Punkte, wo y_true != y_pred (falls vorhanden)
    - max_points: optional Begrenzung pro Modell/Scenario
    """

    def _iter_xai_bundles():
        for scen, payload in (ALL_RESULTS or {}).items():
            if scen == "catastrophic_forgetting":
                xai = (payload.get("xai", {}) if isinstance(payload, dict) else {}) or {}
                for subk in ["A_on_A", "B_on_A", "B_on_B"]:
                    yield f"{scen}::{subk}", xai.get(subk, {}) or {}
            else:
                xai = (payload.get("xai", {}) if isinstance(payload, dict) else {}) or {}
                yield scen, xai

    for scen_name, xai in _iter_xai_bundles():
        base = scen_name.split("::")[0]
        if (not include_baseline) and base == "baseline":
            continue

        print("\n" + "=" * 100)
        print("SCENARIO:", scen_name)
        print("=" * 100)

        for model_key in ["nn", "svm"]:
            model_block = xai.get(model_key, {}) if isinstance(xai, dict) else {}
            anchors = model_block.get("anchors", {}) if isinstance(model_block, dict) else {}

            print("\n" + "-" * 100)
            print("MODEL:", model_key.upper(), "| Anchors")
            print("-" * 100)

            status = anchors.get("status", "n/a") if isinstance(anchors, dict) else "n/a"
            if status != "ok":
                print("status:", status, "| reason:", anchors.get("reason") if isinstance(anchors, dict) else None)
                continue

            per_point = anchors.get("per_point", {}) if isinstance(anchors, dict) else {}
            if not per_point:
                print("Keine per_point Anchors gefunden (per_point ist leer).")
                continue

            shown = 0
            for pid, rec in per_point.items():
                if not isinstance(rec, dict):
                    continue

                y_true = rec.get("y_true")
                y_pred = rec.get("y_pred")

                if only_misclassified and (y_true is not None) and (y_pred is not None) and (y_true == y_pred):
                    continue

                iloc = rec.get("iloc")
                precision = rec.get("precision")
                coverage = rec.get("coverage")
                rules = rec.get("anchor_rules") or []

                print("\n### Point", pid, f"(iloc={iloc})")
                print(f"y_true={y_true} | y_pred={y_pred} | precision={precision} | coverage={coverage}")

                if rules:
                    print("Anchor-Regeln:")
                    for r in rules:
                        print(" -", r)
                else:
                    print("Anchor-Regeln: (keine Regeln gefunden)")

                shown += 1
                if max_points is not None and shown >= int(max_points):
                    print("\n... (abgebrochen nach max_points=%s)" % max_points)
                    break


# Aufruf:
print_anchor_rules_for_error_scenarios(ALL_RESULTS, include_baseline=False, only_misclassified=False)

# Optional (nur falsch klassifizierte):
# print_anchor_rules_for_error_scenarios(ALL_RESULTS, include_baseline=False, only_misclassified=True)



SCENARIO: covariance_shift

----------------------------------------------------------------------------------------------------
MODEL: NN | Anchors
----------------------------------------------------------------------------------------------------

### Point 55838 (iloc=398)
y_true=0 | y_pred=1 | precision=1.0 | coverage=1.0
Anchor-Regeln: (keine Regeln gefunden)

### Point 57436 (iloc=1996)
y_true=1 | y_pred=1 | precision=1.0 | coverage=1.0
Anchor-Regeln: (keine Regeln gefunden)

### Point 56168 (iloc=728)
y_true=1 | y_pred=1 | precision=1.0 | coverage=1.0
Anchor-Regeln: (keine Regeln gefunden)

----------------------------------------------------------------------------------------------------
MODEL: SVM | Anchors
----------------------------------------------------------------------------------------------------

### Point 55838 (iloc=398)
y_true=0 | y_pred=1 | precision=1.0 | coverage=1.0
Anchor-Regeln: (keine Regeln gefunden)

### Point 57436 (iloc=1996)
y_true=1 | y_pred=1 | p